# Phase 3: Explore — PEA Met Network

Exploratory analysis of cleaned station data from the PEA meteorological
network. This notebook loads processed data and produces summary statistics
and basic visual checks.

In [1]:
from pathlib import Path

import pandas as pd

DATA_DIR = Path("../data/processed").resolve()

## Load Processed Data

In [2]:
stations = list(DATA_DIR.glob("*"))
print(f"Found {len(stations)} station directories:")
for s in stations:
    print(f"  {s.name}")

assert len(stations) > 0, "No station data found in data/processed/"

Found 11 station directories:
  stanhope
  north_rustico
  pipeline_manifest.json
  qa_qc_report.csv
  tracadie
  stanley_bridge
  imputation_report.csv
  cavendish
  stanhope_validation.csv
  quality_enforcement_report.csv
  greenwich


In [3]:
# Load the first station that has both daily and hourly data
sample_station = next(
    s for s in stations
    if (s / "station_daily.csv").is_file() and (s / "station_hourly.csv").is_file()
)
daily_path = sample_station / "station_daily.csv"
hourly_path = sample_station / "station_hourly.csv"

daily = pd.read_csv(daily_path, parse_dates=["timestamp_utc"])
hourly = pd.read_csv(hourly_path, parse_dates=["timestamp_utc"])

print(f"Station: {sample_station.name}")
print(f"Daily records: {len(daily)}")
print(f"Hourly records: {len(hourly)}")


Station: stanhope
Daily records: 1006
Hourly records: 24144


## Summary Statistics

In [4]:
numeric_cols = ["air_temperature_c",
             "relative_humidity_pct", "rain_mm",
             "wind_speed_kmh"]
available_cols = [c for c in numeric_cols if c in daily.columns]

print("=== Daily Data Summary ===")
print(daily[available_cols].describe().to_string())

=== Daily Data Summary ===
       air_temperature_c  relative_humidity_pct      rain_mm  wind_speed_kmh
count        1006.000000            1006.000000  1006.000000     1006.000000
mean            8.681146              78.010451     2.703777       14.097942
std             9.202079              10.920697     5.958861        6.046070
min           -14.687500              42.875000     0.000000        3.458333
25%             1.246875              70.427083     0.000000        9.708333
50%             8.543561              78.645833     0.000000       13.291667
75%            16.981250              86.604167     2.275000       16.958333
max            26.733333              99.043478    42.800000       53.625000


In [5]:
print("=== Hourly Data Summary ===")
print(hourly[available_cols].describe().to_string())

=== Hourly Data Summary ===
       air_temperature_c  relative_humidity_pct       rain_mm  wind_speed_kmh
count       24067.000000            24065.00000  24098.000000    24076.000000
mean            8.669406               78.00374      0.112872       14.092956
std             9.575993               14.82777      0.579573        7.805752
min           -21.000000               19.00000      0.000000        0.000000
25%             1.200000               68.00000      0.000000        9.000000
50%             8.500000               80.00000      0.000000       13.000000
75%            16.700000               91.00000      0.000000       18.000000
max            34.600000              100.00000     16.300000       61.000000


## Data Quality Checks

In [6]:
print("=== Missingness (daily) ===")
print(daily.isnull().sum().to_string())

print()
print("=== Missingness (hourly) ===")
print(hourly.isnull().sum().to_string())

=== Missingness (daily) ===
Hmdx                        770
Visibility (km)            1006
Wind Chill                  690
air_temperature_c             0
barometric_pressure_kpa    1006
bui                           0
dc                            0
dew_point_c                   0
dmc                           0
ffmc                          0
fwi                           0
isi                           0
rain_mm                       0
relative_humidity_pct         0
station                       0
timestamp_utc                 0
wind_direction_deg            0
wind_speed_kmh                0

=== Missingness (hourly) ===
Hmdx                       21433
Visibility (km)            24144
Wind Chill                 19287
_quality_flags             23768
air_temperature_c             77
barometric_pressure_kpa    24144
bui                           86
dc                            77
dew_point_c                   68
dmc                           86
ffmc                          86
fwi

In [7]:
print("=== Date Range ===")
ts_min = daily["timestamp_utc"].min()
ts_max = daily["timestamp_utc"].max()
print(f"Daily:  {ts_min} to {ts_max}")
ts_min_h = hourly["timestamp_utc"].min()
ts_max_h = hourly["timestamp_utc"].max()
print(f"Hourly: {ts_min_h} to {ts_max_h}")

dupes_daily = daily.duplicated(subset="timestamp_utc").sum()
dupes_hourly = hourly.duplicated(subset="timestamp_utc").sum()
print(f"Duplicate timestamps — daily: {dupes_daily}, hourly: {dupes_hourly}")


=== Date Range ===
Daily:  2023-04-01 00:00:00 to 2025-12-31 00:00:00
Hourly: 2023-04-01 00:00:00+00:00 to 2025-12-31 23:00:00+00:00
Duplicate timestamps — daily: 0, hourly: 0


## Conclusion

Data loaded successfully. Summary statistics and quality checks printed above.
No errors encountered during exploration.